# Coleta de Dados de Criptomoedas

Este notebook realiza a coleta de dados históricos de Bitcoin, Ethereum, Solana e XRP por meio da API pública da Binance.

Os dados são coletados com periodicidade diária para o período de janeiro de 2021 a dezembro de 2025.

Ativos utilizados:

- BTCUSDT
- ETHUSDT
- SOLUSDT
- XRPUSDT

In [0]:
import requests
import pandas as pd

## Coleta dos dados

A coleta utiliza o endpoint de dados históricos da Binance, com periodicidade diária e limite de até 1000 registros por requisição.

In [0]:
def coletar_cripto(symbol, inicio_ms, fim_ms):
    url = "https://data-api.binance.vision/api/v3/klines"

    params = {
        "symbol": symbol,
        "interval": "1d",
        "startTime": inicio_ms,
        "endTime": fim_ms,
        "limit": 1000
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    if response.status_code != 200:
        print(f"Erro na coleta de {symbol}: {response.status_code}")
        return None

    dados = response.json()

    colunas = [
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_asset_volume",
        "numero_negociacoes",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "ignore"
    ]

    df = pd.DataFrame(
        dados,
        columns=colunas
    )

    colunas_numericas = [
        "open",
        "high",
        "low",
        "close",
        "volume",
        "quote_asset_volume",
        "taker_buy_base_volume",
        "taker_buy_quote_volume"
    ]

    df[colunas_numericas] = df[colunas_numericas].astype(float)

    df["data"] = pd.to_datetime(
        df["open_time"],
        unit="ms"
    )

    df["ativo"] = symbol

    return df


## Coleta do período completo

Como o período analisado ultrapassa o limite de registros de uma única requisição, a coleta é realizada em blocos sucessivos até alcançar a data final definida para o projeto.

In [0]:
def coletar_cripto_periodo_completo(symbol, inicio_ms, fim_ms):
    todos_dados = []
    inicio_atual = inicio_ms

    while inicio_atual <= fim_ms:
        df_parcial = coletar_cripto(
            symbol,
            inicio_atual,
            fim_ms
        )

        if df_parcial is None or df_parcial.empty:
            break

        todos_dados.append(df_parcial)

        ultimo_timestamp = int(df_parcial["open_time"].max())

        inicio_atual = ultimo_timestamp + 86400000

    if not todos_dados:
        return pd.DataFrame()

    df_final = pd.concat(
        todos_dados,
        ignore_index=True
    )

    df_final = df_final.drop_duplicates(
        subset=["open_time"]
    )

    df_final = df_final.sort_values(
        "open_time"
    ).reset_index(drop=True)

    return df_final

In [0]:
inicio_projeto = int(
    pd.Timestamp("2021-01-01", tz="UTC").timestamp() * 1000
)

fim_projeto = int(
    pd.Timestamp("2025-12-31 23:59:59", tz="UTC").timestamp() * 1000
)

print("Início:", inicio_projeto)
print("Fim:", fim_projeto)

## Coleta dos ativos

Nesta etapa são coletados os dados históricos dos quatro ativos definidos para o projeto.

In [0]:
ativos = [
    "BTCUSDT",
    "ETHUSDT",
    "SOLUSDT",
    "XRPUSDT"
]

dfs_criptos = []

for ativo in ativos:
    print(f"Coletando {ativo}...")

    df_ativo = coletar_cripto_periodo_completo(
        ativo,
        inicio_projeto,
        fim_projeto
    )

    dfs_criptos.append(df_ativo)

    print(
        f"{ativo}: {len(df_ativo)} registros | "
        f"{df_ativo['data'].min()} até {df_ativo['data'].max()}"
    )

In [0]:
df_criptos = pd.concat(
    dfs_criptos,
    ignore_index=True
)

print("Total de registros:", len(df_criptos))

display(
    df_criptos[
        ["data", "ativo", "open", "high", "low", "close", "volume"]
    ].head(10)
)

In [0]:
resumo_criptos = (
    df_criptos
    .groupby("ativo")
    .agg(
        quantidade=("data", "count"),
        primeira_data=("data", "min"),
        ultima_data=("data", "max")
    )
    .reset_index()
)

display(resumo_criptos)

## Validação da qualidade dos dados

São verificadas a quantidade de registros, a presença de valores nulos, possíveis duplicidades e a quantidade de observações por ativo.

In [0]:
print("=== VALIDAÇÃO DOS DADOS ===")

print("\nTotal de registros:")
print(len(df_criptos))

print("\nValores nulos:")
print(
    df_criptos[
        ["data", "ativo", "open", "high", "low", "close", "volume"]
    ].isnull().sum()
)

print("\nRegistros duplicados:")
duplicados = df_criptos.duplicated(
    subset=["data", "ativo"]
).sum()

print(duplicados)

print("\nQuantidade por ativo:")
print(
    df_criptos.groupby("ativo").size()
)